# Datacenter Anomaly Detection — EfficientAD

**파이프라인**
1. 영상 → 프레임 추출
2. 구역 분류 (lobby / server_room)
3. 전처리 및 데이터셋 구성
4. EfficientAD-Small 모델 정의
5. 구역별 학습
6. 임계값 캘리브레이션
7. 추론 테스트 및 시각화
8. 모델 저장

---
### 📊 그래프 해석 가이드

**[Cell 9] Loss Curve**
- `Train Loss` : 학습 데이터의 이상 점수 평균. 낮을수록 정상 패턴을 잘 학습한 것
- `Val Loss` : 검증 데이터의 이상 점수 평균
- ✅ 정상: 두 곡선이 함께 수렴 (0.01 이하)
- ⚠️ 과적합: Train만 계속 떨어지고 Val이 올라감 → patience로 자동 중단
- ⚠️ 미수렴: 두 곡선이 진동하거나 평탄 → lr을 낮추거나 데이터 추가

**[Cell 11] Calibration Histogram (2개)**
- 왼쪽 `Pixel Score Dist.` : 정상 이미지 전체 픽셀의 이상 점수 분포
  - 대부분 0 근처에 몰려야 정상
  - 1%ile(초록 점선) = 정규화 하한, 99%ile(빨간 점선) = 정규화 상한
- 오른쪽 `Image Max Score Dist.` : 이미지별 최대 점수 분포
  - 빨간 수직선 = 최종 임계값 (이 선 오른쪽 이미지를 이상으로 판정)
  - 임계값이 분포 오른쪽 꼬리에 위치해야 오탐이 적음
  - 임계값이 분포 한가운데 있으면 → safety_margin 값을 높여야 함

**[Cell 12] Inference Heatmap**
- 파랑→초록→빨강 순으로 이상 점수 높음
- 빨간 바운딩박스 = 임계값 초과 영역
- title 색: 초록=NORMAL, 빨강=ANOMALY
- ✅ 정상: 정상 이미지에서 히트맵이 고르게 파랑
- ⚠️ 문제: 정상 이미지인데 특정 부위만 빨강 → 그 부위가 학습 데이터에 부족한 것

**[Cell 16] Score Timeline**
- x축=프레임, y축=이상 점수
- 빨간 가로선 = 임계값
- 빨간 음영 = 이상으로 판정된 구간
- 정상 영상에서 이상 비율이 5% 이하면 양호

---
### 📈 성능 지표 해석

| 지표 | 의미 | 목표 |
|------|------|------|
| **Val Loss** | 정상 이미지에서의 평균 이상 점수 | 0.02 이하 |
| **False Positive Rate** | 정상인데 이상으로 탐지된 비율 | 5% 이하 |
| **Inference FPS** | 초당 처리 프레임 수 | GPU 기준 30 FPS+ |
| **Threshold** | 임계값. 높을수록 둔감, 낮을수록 민감 | 캘리브레이션 후 수동 미세 조정 |

## Cell 1 — 환경 설정 및 라이브러리 설치

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install torch torchvision opencv-python-headless \
             scikit-learn matplotlib tqdm -q

import os, sys, json, time, shutil, random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# 재현성 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── CUDA 강제 확인 ──────────────────────────────────────────────
# 이 프로젝트는 GPU(CUDA) 전용으로 동작함
# CPU fallback 없음 → GPU 없으면 여기서 중단
assert torch.cuda.is_available(), \
    'CUDA GPU가 필요합니다. GPU 환경에서 실행하세요.'

DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True  # 고정 입력 크기에서 속도 최적화

print(f'Device  : {DEVICE}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch : {torch.__version__}')

## Cell 2 — 경로 및 구역 설정

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 수정 필요: 영상 경로와 영상→구역 매핑
# 구역은 lobby / server_room 두 개만 사용
#   - lobby      : 로비, 복도, 출입문, 소화전
#   - server_room: 서버랙, 창문, 랙 통로, 랙 끝단 — 서버실 전체
# ──────────────────────────────────────────────────────────────────

VIDEO_PATHS = [
    'videos/lobby_01.mp4',
    'videos/server_room_01.mp4',
    # 추가 영상은 여기에 계속 추가
]

# 영상 파일명(확장자 제외) → 구역 매핑
VIDEO_ZONE_MAP = {
    'lobby_01':       'lobby',
    'server_room_01': 'server_room',
    # 추가 영상 매핑
}

# 구역 정의 (두 개 고정)
ZONES = {
    'lobby':       'Lobby, corridor, entrance, fire hydrant panel',
    'server_room': 'Server racks, rack aisles, windows, all areas inside server room',
}

# 출력 디렉토리 생성
ROOT        = Path('efficientad_project')
FRAMES_DIR  = ROOT / 'frames'
DATASET_DIR = ROOT / 'dataset'
MODEL_DIR   = ROOT / 'models'
RESULT_DIR  = ROOT / 'results'

for d in [FRAMES_DIR, DATASET_DIR, MODEL_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for zone in ZONES:
    (DATASET_DIR / zone / 'train').mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / zone / 'test' / 'good').mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / zone / 'test' / 'anomaly').mkdir(parents=True, exist_ok=True)

print('Zones defined:')
for zone, desc in ZONES.items():
    print(f'  [{zone}] {desc}')
print(f'\nProject root: {ROOT.absolute()}')

## Cell 3 — 영상에서 프레임 추출

In [ ]:
def extract_frames(
    video_path: str,
    out_dir: Path,
    sample_fps: float = 2.0,      # 초당 추출 장 수
    blur_threshold: float = 80.0, # Laplacian 분산 기준 (낮을수록 엄격)
    skip_seconds: float = 1.0,    # 영상 앞뒤 스킵 (손 떨림 구간 제거)
) -> list:
    """
    핸드폰 영상 → 선명한 프레임만 추출
    - blur_threshold: 80 기준, 낮추면 더 엄격하게 필터링
    - sample_fps: 2.0 = 초당 2장 (10분 영상 → 약 1,200장)
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f'[ERROR] Cannot open: {video_path}')
        return []

    vid_fps  = cap.get(cv2.CAP_PROP_FPS)
    total_f  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_f / vid_fps
    interval = max(1, int(vid_fps / sample_fps))
    skip_f   = int(skip_seconds * vid_fps)

    video_name = Path(video_path).stem
    saved = []
    prev_gray = None
    frame_idx = 0

    print(f'Video : {video_name}')
    print(f'  Duration={duration:.1f}s  FPS={vid_fps:.1f}  Extract every {interval} frames')

    pbar = tqdm(total=total_f, desc=f'Extracting {video_name}')

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        pbar.update(1)

        # 영상 앞뒤 구간 스킵 (카메라 들고 시작하는 흔들림 제거)
        if frame_idx < skip_f or frame_idx > total_f - skip_f:
            frame_idx += 1
            continue

        # 샘플링 간격 적용
        if frame_idx % interval != 0:
            frame_idx += 1
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 블러 필터: Laplacian 분산이 낮으면 흔들린 프레임
        # blur_threshold=80 → 대부분의 핸드폰 보행 영상에 적합
        if cv2.Laplacian(gray, cv2.CV_64F).var() < blur_threshold:
            frame_idx += 1
            continue

        # 중복 프레임 제거: 직전 프레임과 픽셀 차이 3 미만이면 스킵
        if prev_gray is not None and cv2.absdiff(gray, prev_gray).mean() < 3.0:
            frame_idx += 1
            continue

        # 노출 이상 프레임 제거 (완전 어둡거나 날아간 프레임)
        mean_val = gray.mean()
        if mean_val < 20 or mean_val > 230:
            frame_idx += 1
            continue

        out_path = out_dir / f'{video_name}_f{frame_idx:06d}.jpg'
        cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved.append(out_path)
        prev_gray = gray
        frame_idx += 1

    cap.release()
    pbar.close()
    print(f'  -> Saved: {len(saved)} frames\n')
    return saved


# 모든 영상 처리
all_frames = []
for vp in VIDEO_PATHS:
    if not Path(vp).exists():
        print(f'[WARN] File not found: {vp}')
        continue
    frames = extract_frames(vp, FRAMES_DIR, sample_fps=2.0)
    all_frames.extend(frames)

print(f'Total extracted frames: {len(all_frames)}')

## Cell 4 — 추출된 프레임 샘플 미리보기

In [ ]:
def show_frame_grid(frame_paths: list, n: int = 16, title: str = ''):
    """무작위 N장 미리보기 (영상 추출 품질 육안 확인용)"""
    sample = random.sample(frame_paths, min(n, len(frame_paths)))
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < len(sample):
            img = cv2.imread(str(sample[i]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            # 파일명에서 프레임 번호만 표시
            ax.set_title(Path(sample[i]).stem[-12:], fontsize=7)
        ax.axis('off')

    fig.suptitle(title or f'Sample Frames — {len(sample)} of {len(frame_paths)} total',
                 fontsize=12)
    plt.tight_layout()
    plt.show()


show_frame_grid(all_frames, n=16)

## Cell 5 — 구역 분류

영상 파일명 기반 자동 분류.
미분류 영상이 있으면 Cell 5-B에서 수동 배정.

In [ ]:
def classify_by_filename(frame_paths: list) -> tuple:
    """
    프레임 파일명에서 영상명을 추출하고
    VIDEO_ZONE_MAP으로 구역 배정
    파일명 규칙: {video_name}_f{frame_idx}.jpg
    """
    zone_frames = {z: [] for z in ZONES}
    unclassified = []

    for fp in frame_paths:
        # 마지막 _fXXXXXX 부분 제거해서 영상명 복원
        parts = fp.stem.rsplit('_f', 1)
        video_name = parts[0] if len(parts) == 2 else fp.stem

        zone = VIDEO_ZONE_MAP.get(video_name)
        if zone and zone in ZONES:
            zone_frames[zone].append(fp)
        else:
            unclassified.append(fp)

    print('Auto classification result:')
    for zone, frames in zone_frames.items():
        print(f'  {zone:15s}: {len(frames):5d} frames')
    if unclassified:
        print(f'  [unclassified]: {len(unclassified)} frames -> run Cell 5-B')

    return zone_frames, unclassified


zone_frames, unclassified = classify_by_filename(all_frames)

In [ ]:
# Cell 5-B: 미분류 영상 수동 배정
# unclassified가 비어있으면 이 셀은 스킵해도 됨

if unclassified:
    # 미분류 영상 목록과 대표 프레임 출력
    video_groups = {}
    for fp in unclassified:
        parts = fp.stem.rsplit('_f', 1)
        vname = parts[0] if len(parts) == 2 else fp.stem
        video_groups.setdefault(vname, []).append(fp)

    for vname, frames in video_groups.items():
        # 중간 프레임을 대표 이미지로 표시
        mid_frame = cv2.imread(str(frames[len(frames)//2]))
        mid_frame = cv2.cvtColor(mid_frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(6, 3))
        plt.imshow(mid_frame)
        plt.title(f'{vname}  ({len(frames)} frames)  ->  which zone?')
        plt.axis('off')
        plt.show()

    # 아래 딕셔너리에 영상명: 구역 직접 작성
    # lobby / server_room 중 선택
    manual_map = {
        # 예시:
        # 'tour_03': 'server_room',
        # 'tour_04': 'lobby',
    }

    for vname, zone in manual_map.items():
        if zone in ZONES and vname in video_groups:
            zone_frames[zone].extend(video_groups[vname])
            print(f'Assigned: {vname} -> {zone} ({len(video_groups[vname])} frames)')

print('\nFinal frame count:')
for zone, frames in zone_frames.items():
    print(f'  {zone:15s}: {len(frames):5d} frames')

## Cell 6 — 데이터셋 구성 (Train / Test 분할)

In [ ]:
def build_dataset(zone_frames: dict, test_ratio: float = 0.15) -> dict:
    """
    - Train (85%): EfficientAD 학습용 정상 이미지
    - Test/good (15%): 임계값 캘리브레이션용
    - Test/anomaly: 비워둠 (실제 이상 영상 확보 시 나중에 추가)
    """
    stats = {}
    for zone, frames in zone_frames.items():
        if not frames:
            print(f'[WARN] {zone}: no frames, skipping')
            continue

        random.shuffle(frames)
        n_test  = max(20, int(len(frames) * test_ratio))
        n_train = len(frames) - n_test

        train_dir = DATASET_DIR / zone / 'train'
        test_dir  = DATASET_DIR / zone / 'test' / 'good'

        for fp in tqdm(frames[:n_train], desc=f'{zone} train copy'):
            shutil.copy2(fp, train_dir / fp.name)
        for fp in tqdm(frames[n_train:], desc=f'{zone} test copy'):
            shutil.copy2(fp, test_dir / fp.name)

        stats[zone] = {'train': n_train, 'test': n_test}
        print(f'{zone}: train={n_train}, test={n_test}')

    return stats


dataset_stats = build_dataset(zone_frames, test_ratio=0.15)
print(f'\nDataset saved to: {DATASET_DIR.absolute()}')

## Cell 7 — EfficientAD-Small 모델 정의

In [ ]:
# ── Teacher 네트워크 ─────────────────────────────────────────────
# ImageNet 사전학습 EfficientNet-B4의 앞부분만 사용
# 가중치 고정(requires_grad=False) — 학습되지 않음
# 정상 패턴의 의미론적 특징맵을 추출하는 기준점 역할
class PatchDescriptionNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.efficientnet_b4(weights='IMAGENET1K_V1')
        self.features = nn.Sequential(
            backbone.features[0],
            backbone.features[1],
            backbone.features[2],
            backbone.features[3],
            backbone.features[4],
        )
        # Teacher는 학습 대상이 아님 — 고정
        for p in self.parameters():
            p.requires_grad = False

        # 출력 채널 수 확인 (GPU에서 계산)
        self.to(DEVICE)
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 256, 256, device=DEVICE)
            out = self.features(dummy)
            self.out_channels = out.shape[1]

    def forward(self, x):
        # x는 반드시 CUDA 텐서여야 함
        return self.features(x)


# ── Student 네트워크 ─────────────────────────────────────────────
# Teacher의 출력을 정상 이미지에서만 모방하도록 학습
# 이상 입력에서는 Teacher 출력을 재현하지 못해 오류맵이 커짐
class StudentNetwork(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        mid = in_channels // 2
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid, 3, padding=1),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, out_channels, 1),
        )

    def forward(self, x):
        return self.net(x)


# ── Autoencoder ─────────────────────────────────────────────────
# 원본 이미지를 정상 구조로만 복원하도록 학습
# 이상 이미지는 복원 오류가 크게 발생
class AnomalyAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,   32,  3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32,  64,  3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64,  128, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 128, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 1),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64,  3, stride=2, padding=1, output_padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64,  32,  3, stride=2, padding=1, output_padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32,  3,   3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.bottleneck(self.encoder(x)))


# ── EfficientAD-Small 통합 클래스 ───────────────────────────────
class EfficientAD_Small(nn.Module):
    IMG_SIZE = 256

    def __init__(self):
        super().__init__()
        # 모든 서브모듈을 CUDA로 올림
        self.teacher    = PatchDescriptionNetwork().to(DEVICE)
        ch              = self.teacher.out_channels
        self.student    = StudentNetwork(ch, ch).to(DEVICE)
        self.autoencoder = AnomalyAutoencoder().to(DEVICE)

        # Teacher 정규화 버퍼 (CUDA)
        self.register_buffer('teacher_mean', torch.zeros(1, ch, 1, 1, device=DEVICE))
        self.register_buffer('teacher_std',  torch.ones(1,  ch, 1, 1, device=DEVICE))

        # 이상 점수 정규화 버퍼 (CUDA)
        self.register_buffer('q_low',     torch.tensor(0.0, device=DEVICE))
        self.register_buffer('q_high',    torch.tensor(1.0, device=DEVICE))
        self.register_buffer('threshold', torch.tensor(0.5, device=DEVICE))

        # 전처리 파이프라인 (CPU에서 실행 후 CUDA로 전송)
        self.preprocess = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((self.IMG_SIZE, self.IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])

    def compute_anomaly_map(self, x: torch.Tensor):
        """
        x: CUDA 텐서 (B, 3, H, W) — ImageNet 정규화 완료된 상태
        모든 연산이 CUDA에서 실행됨

        returns:
          combined  (B,1,H,W): 0~1 정규화된 최종 이상 점수맵
          ts_map    (B,1,H,W): Teacher-Student 오류맵 (raw)
          ae_map    (B,1,H,W): Autoencoder 복원 오류맵 (raw)
        """
        assert x.device.type == 'cuda', 'Input must be CUDA tensor'
        B, _, H, W = x.shape

        # Teacher 특징 추출 (고정, no_grad)
        with torch.no_grad():
            t_feat = self.teacher(x)
            t_feat_norm = (t_feat - self.teacher_mean) / (self.teacher_std + 1e-8)

        # Student가 Teacher 특징을 얼마나 잘 재현하는지 오류맵
        s_feat = self.student(t_feat_norm)
        ts_map = ((t_feat_norm - s_feat) ** 2).mean(dim=1, keepdim=True)

        # Autoencoder 복원 오류맵
        ae_out = self.autoencoder(x)
        ae_map = ((x - ae_out) ** 2).mean(dim=1, keepdim=True)

        # Teacher 출력 해상도(32x32) → 원본 해상도(256x256) 업샘플
        ts_map_up = F.interpolate(ts_map, size=(H, W), mode='bilinear', align_corners=False)

        # TS:AE = 6:4 비율로 결합 (TS가 더 정밀한 경향)
        combined = 0.6 * ts_map_up + 0.4 * ae_map

        # 캘리브레이션된 q_low/q_high로 0~1 정규화
        combined = (combined - self.q_low) / (self.q_high - self.q_low + 1e-8)
        combined = combined.clamp(0, 1)

        return combined, ts_map_up, ae_map

    def predict(self, frame_bgr: np.ndarray) -> dict:
        """
        OpenCV BGR 프레임(numpy) 입력 → 이상 탐지 결과 딕셔너리 반환
        전처리(CPU) → CUDA 전송 → 추론(GPU) → 결과(CPU) 순서
        """
        self.eval()
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        # 전처리 후 CUDA로 전송
        x = self.preprocess(frame_rgb).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            anomaly_map, ts_map, ae_map = self.compute_anomaly_map(x)

        # 결과를 numpy로 변환 (CPU로 이동)
        heatmap = anomaly_map.squeeze().cpu().numpy()
        score   = float(anomaly_map.max().cpu())

        return {
            'is_anomaly': score > float(self.threshold.cpu()),
            'score':      score,
            'threshold':  float(self.threshold.cpu()),
            'heatmap':    heatmap,  # 256x256 numpy array
        }


# 초기화 테스트 (GPU 연산 확인)
_model = EfficientAD_Small()
_dummy = torch.zeros(1, 3, 256, 256, device=DEVICE)  # CUDA 텐서
with torch.no_grad():
    _am, _ts, _ae = _model.compute_anomaly_map(_dummy)
print('Model init OK')
print(f'  Teacher out channels : {_model.teacher.out_channels}')
print(f'  Anomaly map shape    : {_am.shape}')
print(f'  All on CUDA          : {_am.device}')
del _model, _dummy, _am, _ts, _ae
torch.cuda.empty_cache()

## Cell 8 — Dataset & DataLoader

In [ ]:
class NormalDataset(Dataset):
    """
    정상 이미지만 담은 데이터셋
    비지도 학습(Unsupervised) 특성상 라벨 없음
    DataLoader의 num_workers 프로세스에서 CPU로 이미지 로드 →
    pin_memory=True로 CUDA 전송 속도 최적화
    """

    # 학습용 증강: 다양한 조명/각도 변화에 강건하게
    AUGMENT = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((288, 288)),
        transforms.RandomCrop(256),
        transforms.RandomHorizontalFlip(p=0.3),
        # 데이터센터 조명 변화(형광등, 역광) 대응
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    # 검증/캘리브레이션용: 증강 없이 고정 크기만
    EVAL = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    def __init__(self, image_dir: Path, augment: bool = True):
        self.paths = sorted(image_dir.glob('*.jpg')) + \
                     sorted(image_dir.glob('*.png'))
        assert len(self.paths) > 0, f'No images in {image_dir}'
        self.transform = self.AUGMENT if augment else self.EVAL

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        # num_workers 프로세스에서 CPU로 로드
        img = cv2.imread(str(self.paths[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img)  # CPU 텐서 반환 → DataLoader가 CUDA로 전송


def make_loaders(zone: str, batch_size: int = 8) -> tuple:
    train_ds = NormalDataset(DATASET_DIR / zone / 'train', augment=True)
    test_ds  = NormalDataset(DATASET_DIR / zone / 'test' / 'good', augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4,
        pin_memory=True,   # CPU → CUDA 전송 속도 향상
        drop_last=True,    # 마지막 불완전 배치 제거 (BatchNorm 안정성)
    )
    test_loader = DataLoader(
        test_ds, batch_size=1, shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    return train_loader, test_loader


print('Dataset summary:')
for zone in ZONES:
    n_train = len(list((DATASET_DIR / zone / 'train').glob('*.jpg')))
    n_test  = len(list((DATASET_DIR / zone / 'test' / 'good').glob('*.jpg')))
    if n_train > 0:
        print(f'  {zone:15s}: train={n_train:5d}  test={n_test:4d}')

## Cell 9 — 학습 루프

### Loss Curve 해석
- `Train Loss` / `Val Loss` 두 곡선이 함께 0.01 수준으로 수렴하면 정상
- `Val Loss`가 올라가기 시작하면 Early Stopping이 자동 종료
- 곡선이 0.05 이상에서 멈추면 데이터 부족 신호 → 영상 더 추가

In [ ]:
def compute_teacher_stats(teacher: nn.Module, loader: DataLoader) -> tuple:
    """
    Teacher 출력의 채널별 평균/표준편차 계산
    이후 Student 입력 정규화에 사용
    모든 연산을 CUDA에서 실행
    """
    teacher.eval()
    feats = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Computing teacher stats'):
            # pin_memory 배치를 non_blocking으로 CUDA 전송
            batch = batch.to(DEVICE, non_blocking=True)
            feats.append(teacher(batch))  # CUDA 텐서 저장

    feats = torch.cat(feats, dim=0)  # (N, C, H, W) CUDA
    mean  = feats.mean(dim=(0, 2, 3), keepdim=True)  # CUDA
    std   = feats.std(dim=(0, 2, 3),  keepdim=True)  # CUDA
    return mean, std


def train_zone(
    zone: str,
    epochs: int     = 200,
    batch_size: int = 8,
    lr: float       = 1e-4,
    patience: int   = 20,
) -> 'EfficientAD_Small':

    print(f'\n{"="*55}')
    print(f'  Training zone: {zone}')
    print(f'{"="*55}')

    train_loader, test_loader = make_loaders(zone, batch_size)

    # 모든 서브모듈이 CUDA에 있는 모델 생성
    model = EfficientAD_Small()

    # Step 1: Teacher 통계 (CUDA 연산)
    print('Step 1/3 — Teacher feature statistics (CUDA)...')
    t_mean, t_std = compute_teacher_stats(model.teacher, train_loader)
    # 버퍼에 복사 (이미 CUDA)
    model.teacher_mean.copy_(t_mean)
    model.teacher_std.copy_(t_std)

    # 학습 대상: Student + Autoencoder만
    params = list(model.student.parameters()) + \
             list(model.autoencoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.01
    )

    print('Step 2/3 — Training (all ops on CUDA)...')
    history    = {'train': [], 'val': []}
    best_loss  = float('inf')
    wait       = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        # ── Train ────────────────────────────────────────────────
        model.student.train()
        model.autoencoder.train()
        model.teacher.eval()  # Teacher는 항상 eval
        t_losses = []

        for batch in train_loader:
            # DataLoader pin_memory → non_blocking으로 빠르게 CUDA 전송
            batch = batch.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()

            anomaly_map, ts_map, ae_map = model.compute_anomaly_map(batch)

            loss_st = ts_map.mean()
            loss_ae = ae_map.mean()

            # Hard negative mining: 오류 큰 상위 10% 패치에 추가 페널티
            n_hard    = max(1, int(ts_map.numel() * 0.1))
            hard_loss = ts_map.reshape(-1).topk(n_hard).values.mean()

            loss = loss_st + loss_ae + 0.5 * hard_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            optimizer.step()
            t_losses.append(loss.item())

        scheduler.step()

        # ── Validation ───────────────────────────────────────────
        model.eval()
        v_losses = []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(DEVICE, non_blocking=True)
                am, _, _ = model.compute_anomaly_map(batch)
                v_losses.append(am.mean().item())

        t_loss = float(np.mean(t_losses))
        v_loss = float(np.mean(v_losses))
        history['train'].append(t_loss)
        history['val'].append(v_loss)

        # Early stopping: best val loss 추적
        if v_loss < best_loss:
            best_loss  = v_loss
            wait       = 0
            # state_dict는 모두 CUDA 텐서 — clone으로 복사 보존
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1

        if epoch % 10 == 0:
            print(f'  Ep {epoch:3d}/{epochs} | '
                  f'train={t_loss:.4f}  val={v_loss:.4f} | '
                  f'lr={scheduler.get_last_lr()[0]:.2e} | '
                  f'patience={wait}/{patience}')

        if wait >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break

    # 최적 가중치 복원 (CUDA 텐서 그대로 로드)
    if best_state:
        model.load_state_dict(best_state)

    # ── Loss Curve 시각화 ─────────────────────────────────────
    # 해석: Train/Val이 함께 수렴 → 정상 | Val만 올라감 → 과적합
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history['train'], label='Train Loss', color='steelblue')
    ax.plot(history['val'],   label='Val Loss',   color='darkorange', linestyle='--')
    ax.axhline(best_loss, color='red', linestyle=':', linewidth=1,
               label=f'Best Val Loss ({best_loss:.4f})')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Anomaly Loss')
    # 목표: 두 곡선이 0.01 이하로 수렴
    ax.set_title(f'Training Curve — {zone}\n'
                 f'Goal: both curves converge below 0.01')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f'{zone}_loss_curve.png', dpi=120)
    plt.show()

    print(f'\nStep 2/3 done — Best val loss: {best_loss:.4f}')
    torch.cuda.empty_cache()  # VRAM 정리
    return model


print('Train function ready. Run Cell 10 to start training.')

## Cell 10 — 구역별 학습 실행

In [ ]:
# 학습 하이퍼파라미터
# - epochs: Early Stopping으로 자동 조기 종료되므로 크게 잡아도 됨
# - patience: val loss가 이 epoch 수만큼 개선 없으면 중단
# - batch_size: VRAM 8GB 기준 8~16 가능
TRAIN_CONFIG = {
    'epochs':     200,
    'batch_size': 8,
    'lr':         1e-4,
    'patience':   20,
}

# 프레임이 20장 이상인 구역만 학습
ZONES_TO_TRAIN = [
    z for z in ZONES
    if len(list((DATASET_DIR / z / 'train').glob('*.jpg'))) >= 20
]
print(f'Zones to train: {ZONES_TO_TRAIN}')

trained_models: dict = {}

for zone in ZONES_TO_TRAIN:
    model = train_zone(zone, **TRAIN_CONFIG)
    trained_models[zone] = model

print('\nAll zones trained.')

## Cell 11 — 임계값 캘리브레이션

### Calibration Histogram 해석
**왼쪽 `Pixel Score Dist.`**
- x축: 픽셀별 이상 점수 (0에 가까울수록 정상)
- 분포가 0 근처에 몰릴수록 모델이 정상을 잘 학습한 것
- 초록 점선(1%ile) = 정규화 하한, 빨강 점선(99%ile) = 정규화 상한

**오른쪽 `Image Max Score Dist.`**
- x축: 이미지당 최대 이상 점수
- 빨간 수직선 = 임계값 (이 값보다 크면 ANOMALY 판정)
- 임계값이 분포 오른쪽 꼬리(상위 5~10%)에 위치해야 정상
- 임계값이 분포 중간에 위치하면 → `safety_margin` 값을 높여서 재실행

In [ ]:
def calibrate_threshold(
    model: 'EfficientAD_Small',
    zone: str,
    percentile: float    = 99.0,  # 정상 이미지 최대 점수의 N%ile을 임계값으로
    safety_margin: float = 1.1,   # 임계값에 곱하는 안전 마진 (오탐 줄임)
) -> float:
    """
    정상 이미지에서의 이상 점수 분포를 계산하고
    상위 percentile * safety_margin 을 임계값으로 설정
    모든 추론은 CUDA에서 실행, 통계 계산만 CPU numpy로
    """
    _, test_loader = make_loaders(zone, batch_size=1)
    model.eval()

    all_pixel_scores = []  # 전체 픽셀 점수 (분포 확인용)
    image_max_scores = []  # 이미지별 최대 점수 (임계값 계산용)

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Calibrating {zone}'):
            batch = batch.to(DEVICE, non_blocking=True)
            am, _, _ = model.compute_anomaly_map(batch)

            # CUDA → CPU → numpy (통계 계산)
            am_np = am.squeeze().cpu().numpy()
            all_pixel_scores.extend(am_np.flatten().tolist())
            image_max_scores.append(float(am_np.max()))

    all_pixel_scores = np.array(all_pixel_scores)
    image_max_scores = np.array(image_max_scores)

    # 정규화 기준값 (0~1 매핑)
    q_low  = float(np.percentile(all_pixel_scores, 1))
    q_high = float(np.percentile(all_pixel_scores, 99))

    # 임계값 = 이미지 최대 점수의 상위 percentile × safety_margin
    threshold = float(np.percentile(image_max_scores, percentile)) * safety_margin
    threshold = min(threshold, q_high)  # 정규화 범위를 초과하지 않도록 클램프

    # 버퍼 업데이트 (CUDA)
    model.q_low.fill_(q_low)
    model.q_high.fill_(q_high)
    model.threshold.fill_(threshold)

    # ── Calibration Histogram ───────────────────────────────────
    # 해석:
    #   왼쪽: 픽셀 점수가 0 근처에 몰릴수록 좋음
    #   오른쪽: 빨간 선(임계값)이 분포 오른쪽 꼬리에 있어야 오탐 적음
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # 왼쪽: 픽셀별 이상 점수 분포
    axes[0].hist(all_pixel_scores, bins=100, color='steelblue', alpha=0.7)
    axes[0].axvline(q_low,  color='green', linestyle='--', linewidth=1.5,
                    label=f'1%ile = {q_low:.3f}  (norm lower bound)')
    axes[0].axvline(q_high, color='red',   linestyle='--', linewidth=1.5,
                    label=f'99%ile = {q_high:.3f}  (norm upper bound)')
    axes[0].set_xlabel('Anomaly Score (per pixel)')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'[{zone}] Pixel Score Dist.\n'
                      f'Goal: peak near 0, thin right tail')
    axes[0].legend(fontsize=8)

    # 오른쪽: 이미지별 최대 점수 분포 + 임계값
    axes[1].hist(image_max_scores, bins=30, color='darkorange', alpha=0.7)
    axes[1].axvline(threshold, color='red', linestyle='-', linewidth=2,
                    label=f'Threshold = {threshold:.3f}')
    axes[1].set_xlabel('Max Anomaly Score (per image)')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'[{zone}] Image Max Score Dist.\n'
                      f'Goal: threshold at far right tail')
    axes[1].legend(fontsize=8)

    plt.suptitle(f'Calibration — {zone}  '
                 f'(percentile={percentile}, margin={safety_margin})',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f'{zone}_calibration.png', dpi=120)
    plt.show()

    print(f'  q_low={q_low:.4f}  q_high={q_high:.4f}  threshold={threshold:.4f}')
    return threshold


thresholds = {}
for zone, model in trained_models.items():
    th = calibrate_threshold(model, zone, percentile=99.0, safety_margin=1.1)
    thresholds[zone] = th

print('\nCalibration done:')
for zone, th in thresholds.items():
    print(f'  {zone:15s}: threshold = {th:.4f}')

## Cell 12 — 추론 테스트 및 히트맵 시각화

### Heatmap 해석
- 컬러맵: 파랑(정상) → 초록 → 빨강(이상)
- 빨간 바운딩박스: 임계값 초과 영역
- title 초록=NORMAL / 빨강=ANOMALY
- **정상 이미지에서 바운딩박스가 많으면** → Cell 13에서 임계값 높이기
- **특정 부위만 반복해서 빨간 히트맵** → 그 부위가 학습 데이터에 부족한 것

In [ ]:
def overlay_heatmap(
    frame_bgr: np.ndarray,
    heatmap: np.ndarray,
    alpha: float     = 0.45,
    threshold: float = 0.5,
) -> np.ndarray:
    """히트맵을 원본 이미지에 오버레이하고 이상 영역 바운딩박스 추가"""
    h, w = frame_bgr.shape[:2]
    hm = cv2.resize(heatmap, (w, h))

    # 0~1 float → 0~255 uint8 변환 후 JET 컬러맵 적용
    hm_uint8 = (hm * 255).clip(0, 255).astype(np.uint8)
    hm_color = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)

    # 임계값 초과 영역 마스크 → 바운딩박스
    mask = (hm > threshold).astype(np.uint8)
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    result = cv2.addWeighted(frame_bgr, 1 - alpha, hm_color, alpha, 0)
    for cnt in contours:
        if cv2.contourArea(cnt) > 300:  # 잡음 수준 컨투어 제거
            x, y, bw, bh = cv2.boundingRect(cnt)
            cv2.rectangle(result, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    return result


def test_on_images(model: 'EfficientAD_Small', zone: str, n_samples: int = 8):
    """
    테스트 이미지 무작위 N장 추론 후 히트맵 시각화
    하단에 오탐율(FP rate) 출력
    """
    test_dir = DATASET_DIR / zone / 'test' / 'good'
    paths    = sorted(test_dir.glob('*.jpg'))
    if not paths:
        print(f'No test images for {zone}')
        return

    samples = random.sample(paths, min(n_samples, len(paths)))
    cols    = 4
    rows    = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
    axes = axes.flatten()

    scores = []
    for i, (fp, ax) in enumerate(zip(samples, axes)):
        frame  = cv2.imread(str(fp))
        result = model.predict(frame)  # GPU 추론 → 결과는 numpy
        scores.append(result['score'])

        vis     = overlay_heatmap(frame, result['heatmap'],
                                  alpha=0.4, threshold=result['threshold'])
        vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

        color = 'red' if result['is_anomaly'] else 'green'
        label = 'ANOMALY' if result['is_anomaly'] else 'NORMAL'
        ax.imshow(vis_rgb)
        ax.set_title(f"{label}  score={result['score']:.3f}",
                     color=color, fontsize=9)
        ax.axis('off')

    for ax in axes[len(samples):]:
        ax.axis('off')

    th         = float(trained_models[zone].threshold.cpu())
    fp_rate    = sum(1 for s in scores if s > th) / len(scores) * 100
    mean_score = float(np.mean(scores))

    # 성능 지표: FP rate 5% 이하이면 양호
    fig.suptitle(
        f'Inference Test — {zone}\n'
        f'Threshold={th:.3f}  Mean Score={mean_score:.3f}  '
        f'False Positive Rate={fp_rate:.1f}%  (target: <5%)',
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f'{zone}_inference_test.png', dpi=120)
    plt.show()

    # 성능 요약 출력
    print(f'{zone} results:')
    print(f'  Mean score     : {mean_score:.4f}')
    print(f'  Max score      : {max(scores):.4f}')
    print(f'  Threshold      : {th:.4f}')
    fp_status = 'OK' if fp_rate < 5 else 'HIGH — raise threshold in Cell 13'
    print(f'  FP rate        : {fp_rate:.1f}%  [{fp_status}]')


print('Running inference test on normal images...')
for zone, model in trained_models.items():
    test_on_images(model, zone, n_samples=8)

## Cell 13 — 임계값 수동 튜닝

- FP rate > 5% → 임계값 **높이기** (오탐 줄임, 미탐 늘어남)
- 실제 장애물을 탐지 못함 → 임계값 **낮추기** (민감도 증가)

In [ ]:
# 현재 임계값 확인
print('Current thresholds:')
for zone, model in trained_models.items():
    print(f'  {zone:15s}: {float(model.threshold.cpu()):.4f}')

print()

# 수동 조정이 필요한 경우 아래 딕셔너리에 값 입력
# 비워두면 캘리브레이션 값 그대로 유지
MANUAL_THRESHOLDS = {
    # 'lobby':       0.65,   # FP rate 높으면 올리기
    # 'server_room': 0.60,
}

for zone, th in MANUAL_THRESHOLDS.items():
    if zone in trained_models:
        # CUDA 버퍼 직접 수정
        trained_models[zone].threshold.fill_(th)
        print(f'Updated: {zone} -> {th}')

print('\nFinal thresholds:')
for zone, model in trained_models.items():
    print(f'  {zone:15s}: {float(model.threshold.cpu()):.4f}')

## Cell 14 — 모델 저장

In [ ]:
def save_models(models: dict) -> dict:
    """
    모델 state_dict + 캘리브레이션 파라미터 저장
    버퍼(CUDA 텐서)는 .cpu()로 변환해서 저장
    """
    meta = {
        'zones':    {},
        'img_size': EfficientAD_Small.IMG_SIZE,
        'created':  time.strftime('%Y-%m-%d %H:%M:%S'),
        'device':   'cuda',
    }

    for zone, model in models.items():
        model.eval()
        save_path = MODEL_DIR / f'{zone}_efficientad.pt'

        # CUDA 버퍼를 cpu()로 변환 후 저장
        torch.save({
            'model_state':  model.state_dict(),
            'threshold':    float(model.threshold.cpu()),
            'q_low':        float(model.q_low.cpu()),
            'q_high':       float(model.q_high.cpu()),
            'teacher_mean': model.teacher_mean.cpu(),
            'teacher_std':  model.teacher_std.cpu(),
            'zone':         zone,
            'img_size':     EfficientAD_Small.IMG_SIZE,
        }, save_path)

        size_mb = save_path.stat().st_size / 1e6
        meta['zones'][zone] = {
            'threshold': float(model.threshold.cpu()),
            'q_low':     float(model.q_low.cpu()),
            'q_high':    float(model.q_high.cpu()),
            'file':      save_path.name,
            'size_mb':   round(size_mb, 2),
        }
        print(f'Saved: {save_path.name}  ({size_mb:.1f} MB)')

    meta_path = MODEL_DIR / 'zone_config.json'
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f'Meta : {meta_path}')
    return meta


meta = save_models(trained_models)

print('\nModel directory:')
for f in sorted(MODEL_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

## Cell 15 — 저장 모델 로드 및 속도 벤치마크

In [ ]:
def load_model(zone: str) -> 'EfficientAD_Small':
    """
    저장된 모델 로드
    모든 텐서를 CUDA로 올려서 반환
    배포 시 이 함수만 사용
    """
    ckpt  = torch.load(
        MODEL_DIR / f'{zone}_efficientad.pt',
        map_location=DEVICE  # 저장 위치와 무관하게 CUDA로 로드
    )
    model = EfficientAD_Small()
    model.load_state_dict(ckpt['model_state'])

    # 캘리브레이션 파라미터 복원 (CUDA 버퍼)
    model.threshold.fill_(ckpt['threshold'])
    model.q_low.fill_(ckpt['q_low'])
    model.q_high.fill_(ckpt['q_high'])
    model.teacher_mean.copy_(ckpt['teacher_mean'].to(DEVICE))
    model.teacher_std.copy_(ckpt['teacher_std'].to(DEVICE))

    model.eval()
    return model


# 로드 테스트
loaded_models = {}
for zone in trained_models:
    loaded_models[zone] = load_model(zone)
    th = float(loaded_models[zone].threshold.cpu())
    print(f'Loaded: {zone:15s}  threshold={th:.4f}')

# ── 추론 속도 벤치마크 ───────────────────────────────────────────
# 실제 로봇 카메라 해상도로 테스트
print('\nInference speed benchmark (CUDA):')
dummy_frame = np.random.randint(0, 255, (1080, 1920, 3), dtype=np.uint8)

for zone, model in loaded_models.items():
    # GPU 워밍업 (첫 실행은 CUDA 커널 컴파일로 느림)
    for _ in range(5):
        model.predict(dummy_frame)
    torch.cuda.synchronize()  # GPU 연산 완료 대기

    N  = 30
    t0 = time.perf_counter()
    for _ in range(N):
        model.predict(dummy_frame)
    torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - t0) / N * 1000

    fps_status = 'OK' if elapsed_ms < 33 else 'SLOW — consider INT8 quantization'
    print(f'  {zone:15s}: {elapsed_ms:.1f} ms/frame  '
          f'({1000/elapsed_ms:.1f} FPS)  [{fps_status}]')

## Cell 16 — 동영상 추론 및 Score Timeline

### Score Timeline 해석
- x축: 프레임 번호 (시간 흐름)
- y축: 해당 프레임의 이상 점수
- 빨간 가로선: 임계값
- 빨간 음영: 이상 판정 구간
- **정상 영상에서 빨간 음영 비율 < 5%** 이면 양호
- 특정 시점에 급격한 스파이크 → 그 구간의 영상 프레임을 직접 확인 권장

In [ ]:
def run_video_inference(
    video_path: str,
    zone: str,
    model: 'EfficientAD_Small',
    output_path: str = None,
    max_frames: int  = 300,
):
    """
    검증용 영상 전체 추론
    추론은 모두 CUDA에서 실행
    결과 영상 저장 + Score Timeline 그래프
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if output_path is None:
        output_path = str(RESULT_DIR / f'{zone}_inference_video.mp4')

    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps, (w, h)
    )

    scores      = []
    frame_count = 0
    th          = float(model.threshold.cpu())
    pbar        = tqdm(total=max_frames, desc=f'Inference: {zone}')

    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # GPU 추론
        result = model.predict(frame)
        scores.append(result['score'])

        # 히트맵 오버레이
        vis   = overlay_heatmap(frame, result['heatmap'], alpha=0.4, threshold=th)
        color = (0, 0, 255) if result['is_anomaly'] else (0, 200, 0)
        label = f"{'ANOMALY' if result['is_anomaly'] else 'NORMAL'} "\
                f"score={result['score']:.3f}  th={th:.3f}"
        cv2.putText(vis, label, (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
        cv2.putText(vis, f'Zone: {zone}  Frame: {frame_count}',
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 1)
        writer.write(vis)
        frame_count += 1
        pbar.update(1)

    cap.release()
    writer.release()
    pbar.close()

    # ── Score Timeline ────────────────────────────────────────────
    # 빨간 음영 비율 5% 이하면 양호
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(scores, linewidth=0.8, color='steelblue', label='Anomaly Score')
    ax.axhline(th, color='red', linestyle='--', linewidth=1.5,
               label=f'Threshold ({th:.3f})')
    ax.fill_between(
        range(len(scores)), scores, th,
        where=[s > th for s in scores],
        alpha=0.35, color='red', label='Anomaly region'
    )
    ax.set_xlabel('Frame')
    ax.set_ylabel('Anomaly Score')

    anomaly_ratio = sum(1 for s in scores if s > th) / len(scores) * 100
    status = 'OK' if anomaly_ratio < 5 else 'HIGH — check video or raise threshold'
    ax.set_title(
        f'Score Timeline — {zone}\n'
        f'Anomaly rate={anomaly_ratio:.1f}%  [{status}]  '
        f'Mean={np.mean(scores):.3f}  Max={np.max(scores):.3f}'
    )
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f'{zone}_score_timeline.png', dpi=120)
    plt.show()

    print(f'Output video : {output_path}')
    print(f'Anomaly rate : {anomaly_ratio:.1f}%  (target: <5% for normal video)')


# 사용 예시 — 학습에 사용하지 않은 새 영상으로 검증
# run_video_inference(
#     video_path='videos/validation_server_room.mp4',
#     zone='server_room',
#     model=loaded_models['server_room'],
#     max_frames=500,
# )

print('Cell ready. Uncomment run_video_inference() above and run.')